# 04 EDA Open Source

Generates source coverage, temporal coverage, missingness, and numeric correlation summaries for public features only.

In [15]:
import importlib.util  # Import modules directly from file paths.
import sys  # Register imported modules for cross-module imports.
from pathlib import Path  # Work with filesystem paths.

PROJECT_ROOT = Path("/content/HDX-sources-and-more-API-connection")  # Set repo root.
SRC_DIR = PROJECT_ROOT / "src"  # Set src folder.

def load_module(module_name, module_path):  # Load one Python file as a module.
    spec = importlib.util.spec_from_file_location(module_name, module_path)  # Create import spec.
    module = importlib.util.module_from_spec(spec)  # Create module object.
    sys.modules[module_name] = module  # Register module so other files can import it.
    spec.loader.exec_module(module)  # Execute module code.
    return module  # Return loaded module.

paths = load_module("paths", SRC_DIR / "paths.py")  # Load paths.py first.
cleaning = load_module("cleaning", SRC_DIR / "cleaning.py")  # Load cleaning.py second.
feature_assembly = load_module("feature_assembly", SRC_DIR / "feature_assembly.py")  # Load feature assembly third.

CLEAN_DIR = paths.CLEAN_DIR  # Get cleaned output folder.
MODEL_FEATURES_DIR = paths.MODEL_FEATURES_DIR  # Get model feature folder.
BASE_FEATURE_TABLE_PATH = MODEL_FEATURES_DIR / "base_feature_table_country_month_year.csv"  # Set base table path.

print("Direct module imports worked.")  # Confirm imports.
print(f"Cleaned CSV count before 03c: {len(list(CLEAN_DIR.glob('*.csv')))}")  # Show cleaned file count.

if not list(CLEAN_DIR.glob("*.csv")):  # Regenerate cleaned files if missing.
    summary_report, reject_rows = cleaning.clean_silver_directory()  # Run 03c cleaning.

assembled_features, assembly_report, feature_catalog = feature_assembly.assemble_feature_table()  # Run 03d assembly.

print(f"Base feature table exists: {BASE_FEATURE_TABLE_PATH.exists()}")  # Confirm output exists.
print(f"Rows: {len(assembled_features):,}")  # Show row count.
print(f"Columns: {len(assembled_features.columns):,}")  # Show column count.

Direct module imports worked.
Cleaned CSV count before 03c: 0
Base sources merged: 7
Wide sources deferred: ['clean_whs2026_country_month.csv', 'clean_worldriskindex_country_month.csv']
Base feature rows: 46,522
Base feature columns: 24
Wrote: /content/HDX-sources-and-more-API-connection/outputs/model_features/base_feature_table_country_month_year.csv
Base feature table exists: True
Rows: 46,522
Columns: 24


In [16]:
import pandas as pd  # Load pandas for EDA.

base_features = pd.read_csv(BASE_FEATURE_TABLE_PATH, low_memory=False)  # Read base feature table.

print(f"Rows: {len(base_features):,}")  # Show row count.
print(f"Columns: {len(base_features.columns):,}")  # Show column count.
print(f"Duplicate country-month rows: {base_features.duplicated(['iso3', 'country', 'year', 'month']).sum()}")  # Check key uniqueness.

base_features.head()  # Preview table.

Rows: 46,522
Columns: 24
Duplicate country-month rows: 0


,iso3,country,year,month,civilian_targeting_events,civilian_targeting_fatalities,demonstration_events_events,gdacs_event_count,gdacs_max_severity_value,gdacs_mean_severity_value,...,inform_risk_annual_carried_monthly,political_violence_events,political_violence_fatalities,views_conflict_forecasts_views_main_mean,views_conflict_forecasts_views_main_dich,views_conflict_forecasts_views_main_mean_ln,who_covid_covid_new_cases,who_covid_covid_new_deaths,who_covid_covid_cumulative_cases,who_covid_covid_cumulative_deaths
0,AD,Andorra,2020,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
1,AD,Andorra,2020,2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
2,AD,Andorra,2020,3,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,376.0,12.0,376.0,12.0
3,AD,Andorra,2020,4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,368.0,30.0,744.0,42.0
4,AD,Andorra,2020,5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,20.0,9.0,764.0,51.0


In [17]:
missingness = (  # Build missingness summary.
    base_features
    .isna()
    .mean()
    .reset_index(name="missing_rate")
    .rename(columns={"index": "column"})
    .sort_values("missing_rate", ascending=False)
)

missingness.head(25)  # Show most-missing columns.

,column,missing_rate
7,gdacs_event_count,0.998753
8,gdacs_max_severity_value,0.998753
9,gdacs_mean_severity_value,0.998753
18,views_conflict_forecasts_views_main_dich,0.978505
17,views_conflict_forecasts_views_main_mean,0.978505
19,views_conflict_forecasts_views_main_mean_ln,0.978505
4,civilian_targeting_events,0.869739
6,demonstration_events_events,0.869739
5,civilian_targeting_fatalities,0.869739
16,political_violence_fatalities,0.869739


In [18]:
coverage = base_features.groupby("year").agg(  # Summarize yearly coverage.
    country_count=("iso3", "nunique"),
    row_count=("iso3", "size"),
).reset_index()

coverage  # Display year-level coverage.

,year,country_count,row_count
0,1997,13,156
1,1998,13,156
2,1999,13,156
3,2000,13,156
4,2001,13,156
5,2002,13,156
6,2003,13,156
7,2004,13,156
8,2005,13,156
9,2006,13,156


In [19]:
coverage.head(25)

,year,country_count,row_count
0,1997,13,156
1,1998,13,156
2,1999,13,156
3,2000,13,156
4,2001,13,156
5,2002,13,156
6,2003,13,156
7,2004,13,156
8,2005,13,156
9,2006,13,156


In [21]:
numeric_features = base_features.drop(
    columns=["year", "month"],
    errors="ignore"
).select_dtypes(include="number")

correlations = numeric_features.corr().stack().reset_index()
correlations.columns = ["feature_1", "feature_2", "correlation"]
correlations = correlations[correlations["feature_1"] < correlations["feature_2"]]

correlations.sort_values(
    "correlation",
    key=lambda s: s.abs(),
    ascending=False
).head(25)

,feature_1,feature_2,correlation
79,gdacs_mean_severity_value,inform_risk_inform_inform,-1.000000
64,gdacs_max_severity_value,inform_risk_inform_inform,-1.000000
63,gdacs_max_severity_value,inform_risk_inform_ha,1.000000
65,gdacs_max_severity_value,inform_risk_inform_vu,-1.000000
80,gdacs_mean_severity_value,inform_risk_inform_vu,-1.000000
78,gdacs_mean_severity_value,inform_risk_inform_ha,1.000000
62,gdacs_max_severity_value,inform_risk_inform_cc,-1.000000
77,gdacs_mean_severity_value,inform_risk_inform_cc,-1.000000
61,gdacs_max_severity_value,gdacs_mean_severity_value,0.994506
142,political_violence_events,views_conflict_forecasts_views_main_mean,0.954275


In [22]:
correlations.sort_values('correlation', key=lambda s: s.abs(), ascending=False).head(25) if not correlations.empty else correlations

,feature_1,feature_2,correlation
79,gdacs_mean_severity_value,inform_risk_inform_inform,-1.000000
64,gdacs_max_severity_value,inform_risk_inform_inform,-1.000000
63,gdacs_max_severity_value,inform_risk_inform_ha,1.000000
65,gdacs_max_severity_value,inform_risk_inform_vu,-1.000000
80,gdacs_mean_severity_value,inform_risk_inform_vu,-1.000000
78,gdacs_mean_severity_value,inform_risk_inform_ha,1.000000
62,gdacs_max_severity_value,inform_risk_inform_cc,-1.000000
77,gdacs_mean_severity_value,inform_risk_inform_cc,-1.000000
61,gdacs_max_severity_value,gdacs_mean_severity_value,0.994506
142,political_violence_events,views_conflict_forecasts_views_main_mean,0.954275
